In [ ]:
from train import train_model
import load_data_jax_metrics
from train_memory_efficient import train_model, Trainer
import importlib 
import numpy as np

In [ ]:
#dataset_list=['frequencies_low','frequencies_high','frequencies_noisy','frequencies_small','suma','mackey','legendre3','btc','sp500'] 
#first with only one dataset
dataset_list=['mackey','frequencies_high']
sample_size_list=[4,5] #the sample size for each dataset, for euro and legendre this is 5.  For mackey is 4

#qubits_list=[16,20,24,32] #in the paper we utilize 2,4,6 or 8 
qubits_list=[2,4,6,8,10,12,14]
hidden_list=[16]#[16,32,48,64]
key_list=[1,2,3,4,5,6] #the key for the random number generator
kernel_size=2
architecture='super_parallel' #options are no_reupload, parallel and super_parallel
#point_list=[268,518,768,1018,1268,1518,1768,2018,2268,3518,2768,3018,3268,3518,3768,4018]
point_list=[1000]
 # if you want to work with parallel ansatz indicate the number of layers in the loop below in variable n_layers
model="QLSTM" #options are LSTM, QLSTM
convergence=False

In [ ]:
times=np.zeros((int(len(point_list)),int(len(key_list))))
for p in range(len(point_list)):  
    points=point_list[p] 
    for k in range(len(key_list)):
        key=key_list[k]
        for d in range(len(dataset_list)):
            dataset=dataset_list[d]
            seq_len=sample_size_list[d]
            
            X_train,Y_train,X_test,Y_test,trainloader,testloader,data,features=load_data_jax_metrics.data(dataset,points)
            print(features)
            target_size=1
            for q in range(len(qubits_list)):
                n_qubits=qubits_list[q]
                if architecture=='super_parallel':
                    n_layers=n_qubits//kernel_size
                elif architecture=='parallel' or architecture=='no_reupload':
                    n_layers=4
                for h in range(len(hidden_list)):
                    concat_size=hidden_list[h]
                    #run_Name=dataset+ansatz+str(out_channels)+str(n_layers)+str(architecture)+str(key)
                    run_name=dataset+str(concat_size)+str(n_qubits)+str(key)+str('small8Q25')
                    train_model(X_train,Y_train,X_test,Y_test,trainloader,testloader,data,run_name,dataset, seq_len,n_layers,n_qubits,concat_size,target_size,key,model,convergence,monitor_memory=True)

In [ ]:
import os, re, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt

def plot_gpu_peak_vs_qubits(dataset_dir: str,
                            valid_qubits=(2,4,6,8,10,12,14,16,20),
                            out_dir=None,
                            also_per_phase=True):
    """
    Busca dataset_dir/peaks_*.csv, agrupa por qubits y seed, y grafica:
      - GPU peak total por qubits (media ± std) con puntos por seed
      - (opcional) GPU peak por fase (init, compile, last_epoch) por qubits
    """
    if out_dir is None:
        out_dir = os.path.join(dataset_dir, "figs_profile")
    os.makedirs(out_dir, exist_ok=True)

    # patrón para recuperar qubits/seed desde run_name dentro del filename
    # acepta nombres como: peaks_sp500_16205small8_Q25.csv  o peaks_16... igual
    # (realmente solo necesitamos el bloque después de 16 y antes de 'small8')
    VALID_Q = sorted([str(q) for q in valid_qubits], key=len, reverse=True)
    pat_digits = re.compile(r"16(\d+?)small8", re.IGNORECASE)

    def parse_q_seed_from_name(fname:str):
        m = pat_digits.search(fname)
        if not m:
            return None, None
        digits = m.group(1)
        for qstr in VALID_Q:          # prueba prefijo largo primero (p.ej. '20' antes que '2')
            if digits.startswith(qstr):
                seed_part = digits[len(qstr):]
                return int(qstr), int(seed_part) if seed_part else 0
        # fallback
        return int(digits[0]), int(digits[1:] or 0)

    # carga todos los peaks_*.csv
    paths = glob.glob(os.path.join(dataset_dir, "peaks_*.csv"))
    if not paths:
        print(f"No se encontraron peaks_*.csv en {dataset_dir}")
        return

    rows = []
    for p in paths:
        q, seed = parse_q_seed_from_name(os.path.basename(p))
        if q is None:  # intenta extraer también del nombre de la carpeta (por si cambió el patrón)
            q, seed = parse_q_seed_from_name(p)
        try:
            df = pd.read_csv(p)
        except Exception as e:
            print(f"⚠️ No pude leer {p}: {e}")
            continue
        df["file"] = os.path.basename(p)
        df["qubits"] = q
        df["seed"] = seed
        rows.append(df)

    if not rows:
        print("No se pudieron parsear archivos con qubits/seed.")
        return

    peaks = pd.concat(rows, ignore_index=True)

    # ---- elección de qué pico usar para la figura principal ----
    # Opción 1: pico global por archivo (máximo sobre todos los segmentos)
    per_file_peak = (peaks.groupby(["file","qubits","seed"])["gpu_peak_mib"]
                           .max().reset_index(name="gpu_peak_mib"))

    # Agregado por qubits: media y std sobre semillas
    agg = (per_file_peak.groupby("qubits")["gpu_peak_mib"]
                      .agg(["mean","std","count"])
                      .reset_index()
                      .sort_values("qubits"))

    # Figura principal: barras media ± std + puntos por seed
    plt.figure()
    x = agg["qubits"].values
    y = agg["mean"].values
    yerr = agg["std"].values
    plt.bar(x, y, yerr=yerr, capsize=4, alpha=0.6)
    # puntos individuales (jitter leve)
    for q in sorted(per_file_peak["qubits"].unique()):
        sub = per_file_peak[per_file_peak["qubits"]==q]["gpu_peak_mib"].values
        if len(sub):
            xp = np.full_like(sub, q, dtype=float) + (np.random.rand(len(sub))-0.5)*0.25
            plt.scatter(xp, sub, s=18)
    plt.xlabel("Qubits")
    plt.ylabel("GPU peak (MiB)")
    plt.title(f"Pico de memoria GPU vs qubits\n({os.path.basename(dataset_dir)})")
    plt.xticks(x, x)
    plt.tight_layout()
    out_main = os.path.join(out_dir, f"{os.path.basename(dataset_dir)}_gpu_peak_vs_qubits.png")
    plt.savefig(out_main, dpi=180)
    plt.close()
    print(f"→ Guardado {out_main}")

    # CSV agregado para tabla del paper
    out_csv = os.path.join(out_dir, f"{os.path.basename(dataset_dir)}_gpu_peak_vs_qubits.csv")
    agg.to_csv(out_csv, index=False)
    print(f"→ Guardado {out_csv}")

    if also_per_phase:
        # Para cada fase (segment), media ± std por qubits
        wanted = ["init_after_trainer", "after_warmup_compile", "last_epoch"]
        for seg in wanted:
            seg_df = peaks[peaks["segment"]==seg]
            if seg_df.empty:
                continue
            per_file_seg = (seg_df.groupby(["file","qubits","seed"])["gpu_peak_mib"]
                                   .max().reset_index(name="gpu_peak_mib"))
            agg_seg = (per_file_seg.groupby("qubits")["gpu_peak_mib"]
                                   .agg(["mean","std","count"]).reset_index()
                                   .sort_values("qubits"))
            plt.figure()
            x = agg_seg["qubits"].values
            y = agg_seg["mean"].values
            yerr = agg_seg["std"].values
            plt.bar(x, y, yerr=yerr, capsize=4, alpha=0.7)
            plt.xlabel("Qubits")
            plt.ylabel("GPU peak (MiB)")
            plt.title(f"Pico GPU por fase: {seg}\n({os.path.basename(dataset_dir)})")
            plt.xticks(x, x)
            plt.tight_layout()
            outp = os.path.join(out_dir, f"{os.path.basename(dataset_dir)}_gpu_peak_{seg}.png")
            plt.savefig(outp, dpi=180)
            plt.close()
            print(f"→ Guardado {outp}")


In [ ]:
# Para tu carpeta de dataset (donde están Loss* y peaks_*.csv)
plot_gpu_peak_vs_qubits("sp500")                 # o "frequencies_high"
plot_gpu_peak_vs_qubits("frequencies_high")
